In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import ticker

from waggon.optim import Optimiser
from waggon import functions as f
from waggon.surrogates import GP
from waggon.acquisitions import EI

In [ ]:
class SurrogateOptimiser(Optimiser):
    
    def __init__(self, func, surr, acqf, **kwargs):
        kwargs['func'] = func
        super(SurrogateOptimiser, self).__init__(**kwargs)
        
        self.func               = func
        self.surr               = surr
        self.acqf               = acqf
        
        self.n_candidates       = kwargs['n_candidates'] if 'n_candidates' in kwargs else 10201
        self.num_opt_start      = kwargs['num_opt_start'] if 'num_opt_start' in kwargs else 'grid'
        self.num_opt_disp       = kwargs['num_opt_disp'] if 'num_opt_disp' in kwargs else False
        self.num_opt_tol        = kwargs['num_opt_tol'] if 'num_opt_tol' in kwargs else 1e-8
        self.num_opt_candidates = kwargs['num_opt_candidates'] if 'num_opt_candidates' in kwargs else 128
        self.surr.verbose   = self.verbose
    
    def sample(self):
        # YOUR SAMPLING ALGORITHM
        # self.acqf is the acquisition function
        # which we should take as log p(x)
        # do NOT forget to multiply by -1, i.e. to take it with a minus
        pass

    def predict(self, X, y):
        
        self.surr.fit(X, y)
        
        self.acqf.y = y.reshape(y.shape[0]//self.func.n_obs, self.func.n_obs)
        self.acqf.conds = X[::self.func.n_obs]
        self.acqf.model = self.surr
        
        next_x = self.sample()
        
        return torch.Tensor([next_x])
    

    def plot_iteration_results(self, X, next_x):
        '''
        For surrogate optimiser only.
        '''
        if self.func.dim == 1:
            inter_conds = np.linspace(self.func.domain[:, 0], self.func.domain[:, 1], 121)
        else:
            inter_conds = self.create_candidates(N=10201)
        
        # transform = lambda x: np.exp(x) if self.func.log_transform else lambda x: x

        mu, _ = self.surr.predict(inter_conds)
        mu = mu
        y_true = self.func(inter_conds)
        
        if self.func.dim == 1:
            plt.plot(inter_conds, mu, label=f'Pred: {np.mean((y_true - mu)**2):.2f}', c='cornflowerblue')
            plt.plot(inter_conds, y_true, label='True', c='orange')
            # plt.scatter(X, y, c='black')
            plt.legend()
        else:
            
            y_true, mu = y_true.reshape(101, 101), mu.reshape(101, 101)
            mse = (y_true - mu)**2

            ei = self.acqf(inter_conds).reshape(101, 101)
            x_pred = inter_conds[np.argmin(ei)]
            
            def single_2d_plot(axis, f, title):
                plt.subplot(axis)
                colormap = plt.contourf(f, locator=ticker.LinearLocator(), extent=self.func.domain.flatten(), vmin=np.min(f), vmax=np.max(f))
                plt.scatter(X[:, 0], X[:, 1], color='black')
                for gb in self.func.glob_min:
                    plt.scatter(gb[0], gb[1], color='red', marker='*')
                plt.scatter(x_pred[0], x_pred[1], color='cyan', marker='*')
                plt.scatter(next_x[0], next_x[1], color='magenta', marker='*')
                plt.title(title)
                plt.colorbar(colormap)
            
            plt.figure(figsize=(24, 6))

            single_2d_plot(141, y_true, 'True Function')
            single_2d_plot(142, mu, 'Estimated Function')
            single_2d_plot(143, mse, 'MSE')
            single_2d_plot(144, ei, 'EI')
            
            plt.tight_layout()
        
        plt.show()


In [ ]:
opt = SurrogateOptimiser( # возьмем алгоритм суррогатной оптимизации а.к.а. Байесовская оптимизация и не только
            func=f.three_hump_camel(), # передадим ему ф-ю черного ящика
            surr=GP(), # суррогатную модель
            acqf=EI(), # ф-ю выбору
            seed=2, # сид фиксирует позицию изначальных 2 * d точек - нужно для воспроизводимости
            max_iter=10, # максимальное кол-во итерации
            plot_results=True # рисует ф-ию, аппроскимацию, ско аппрексимации, ф-ю выбора; черные точки - известные тру значения,
            # ЗВЕЗДЫ: красная - глобальный оптимум, голубая - оценка оптимума ф-ии выбора, розовая - выбранная точка
)
_ = opt.optimise()